# Exploratory Data Analysis CNEFE on Splink

## Setup

In [1]:

import getpass

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from splink.backends.spark import similarity_jar_location
from splink import SparkAPI
from pathlib import Path
import warnings
import os

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.home() / "projects" / "brazilian_address_linkage"
DATA_ROOT = Path(f"/media/{getpass.getuser()}/Seagate Portable Drive/Datalake/")
UF = "RJ"

conf = SparkConf()
conf.set("spark.driver.memory", "12g")
conf.set("spark.master", "local[6]")
path = similarity_jar_location()
conf.set("spark.jars", path)

sc = SparkContext.getOrCreate(conf=conf)

spark = SparkSession(sc)
spark.sparkContext.setCheckpointDir("./tmp_checkpoints")
db_api = SparkAPI(spark_session=spark)
cnefe_df = spark.read.parquet(
    os.path.join(DATA_ROOT, "silver/cnefe/cleaned_cnefe_addresses.parquet")
)
cnefe_rj_df = cnefe_df.filter(cnefe_df.UF == UF)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/10 08:42:10 WARN Utils: Your hostname, lucas-vital-Q570M-D3H, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp8s0)
26/09/10 08:42:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/09/10 08:42:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
from pyspark.sql import functions as F

cnefe_rj_df = cnefe_rj_df.withColumn("unique_id", F.monotonically_increasing_id())

In [3]:
total_rows = cnefe_rj_df.count()
print(f"Quantidade registros {UF}: {total_rows:,d}")

Quantidade registros RJ: 8,962,200


## Completness analysis

In [4]:
from splink.exploratory import completeness_chart

completeness_chart(cnefe_rj_df, db_api=db_api)

26/09/10 08:42:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/10 08:43:08 WARN DAGScheduler: Broadcasting large task binary with size 1669.2 KiB


alt.LayerChart(...)

In [5]:
LINKAGE_COLS = [
    "COD_UNICO_ENDERECO",
    "COD_MUNICIPIO",
    "COD_DISTRITO",
    "COD_SUBDISTRITO",
    "COD_SETOR",
    "NUM_QUADRA",
    "NUM_FACE",
    "CEP",
    "DSC_LOCALIDADE",
    "NOM_TIPO_SEGLOGR",
    "NOM_SEGLOGR",
    "NUM_ENDERECO",
    "LATITUDE",
    "LONGITUDE",
    "NV_GEO_COORD",
    "COD_ESPECIE",
    "UF",
    "COD_TIPO_ESPECI",
    "ENDERECO_COMPLETO",
    "NOM_SEGLOGR_phon",
    "DSC_LOCALIDADE_phon",
    "ENDERECO_COMPLETO_phon",
]

## Profiling Columns

In [6]:
from splink.exploratory import profile_columns

profile_columns(cnefe_rj_df, column_expressions=LINKAGE_COLS, db_api=db_api)


26/09/10 08:44:22 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/09/10 08:44:25 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/09/10 08:44:25 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/09/10 08:44:25 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/09/10 08:44:26 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/09/10 08:45:56 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/09/10 08:45:56 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/09/10 08:45:56 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/09/10 08:45:56 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/09/10 08:45:56 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/09/10 08:45:56 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/09/10 08:45:56 WARN DAGScheduler: Broadcasting larg

alt.VConcatChart(...)

## Blocking Analaysis

In [7]:
from splink import block_on
from splink.blocking_analysis import (
    count_comparisons_from_blocking_rule,
    cumulative_comparisons_to_be_scored_from_blocking_rules_data,
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
    n_largest_blocks,
)


blocking_rules = [
    block_on("ENDERECO_COMPLETO_phon", "COD_MUNICIPIO"),
    block_on("ENDERECO_COMPLETO_phon", "DSC_LOCALIDADE_phon"),
    block_on("ENDERECO_COMPLETO_phon", "CEP"),
    block_on("ENDERECO_COMPLETO_phon", "CEP","COD_SETOR"),
    block_on("ENDERECO_COMPLETO_phon", "CEP"),
    block_on("ENDERECO_COMPLETO_phon", "NUM_ENDERECO"),
    block_on("ENDERECO_COMPLETO_phon","NUM_ENDERECO","COD_MUNICIPIO"),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon"),
    block_on("ENDERECO_COMPLETO_phon","DSC_LOCALIDADE_phon","COD_MUNICIPIO"),
    block_on("ENDERECO_COMPLETO_phon", "COD_MUNICIPIO","COD_SETOR"),
    block_on("ENDERECO_COMPLETO_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon"),
    block_on("ENDERECO_COMPLETO_phon","COD_MUNICIPIO", "DSC_LOCALIDADE_phon", "COD_ESPECIE"),
    block_on("COD_UNICO_ENDERECO")
]

## Count number of comparisons generated per blocking rules

In [8]:
import pandas as pd

comparisons_df = pd.DataFrame(
    [
        count_comparisons_from_blocking_rule(
            table_or_tables=cnefe_rj_df,
            blocking_rule=br,
            db_api=db_api,
            link_type="dedupe_only",
            max_rows_limit=1_000_000_000
        )
        for br in blocking_rules
    ]
)
comparisons_df

,number_of_comparisons_generated_pre_filter_conditions,number_of_comparisons_to_be_scored_post_filter_conditions,filter_conditions_identified,equi_join_conditions_identified,link_type_join_condition
0,83030600,37034200,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
1,41295544,16166672,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
2,48787128,19912464,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
3,29086038,10061919,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
4,48787128,19912464,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
5,721225550,356131675,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
6,83030594,37034197,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
7,41295544,16166672,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
8,40365758,15701779,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`
9,30471218,10754509,,l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMPL...,where l.`unique_id` < r.`unique_id`


### Cumulative analysis on blocking rules

In [9]:
cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=cnefe_rj_df,
    blocking_rules=blocking_rules[::-1],
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

In [10]:
cum_comparisons_df = cumulative_comparisons_to_be_scored_from_blocking_rules_data(
    table_or_tables=cnefe_rj_df,
    blocking_rules=blocking_rules[::-1],
    db_api=db_api,
    link_type="dedupe_only",
)
cum_comparisons_df

,blocking_rule,row_count,cumulative_rows,cartesian,match_key,start
0,l.`COD_UNICO_ENDERECO` = r.`COD_UNICO_ENDERECO`,75587,75587,40160509938900,0,0
1,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,9794236,9869823,40160509938900,1,75587
2,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,5831956,15701779,40160509938900,2,9869823
3,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,830057,16531836,40160509938900,3,15701779
4,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,0,16531836,40160509938900,4,16531836
5,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,464893,16996729,40160509938900,5,16531836
6,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,20502361,37499090,40160509938900,6,16996729
7,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,318632585,356131675,40160509938900,7,37499090
8,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,0,356131675,40160509938900,8,356131675
9,(l.`ENDERECO_COMPLETO_phon` = r.`ENDERECO_COMP...,0,356131675,40160509938900,9,356131675
